# 🚀 Project Setup — Educational Quickstart Blueprint

**AI Learning Playground — Environment Configuration & Model Download**

Run this notebook **once** before opening any of the four starter notebooks.
It installs all dependencies, validates your GPU, authenticates with Hugging Face,
and downloads the quantized GGUF models used across all four blueprints.

---

## What This Notebook Does

| Cell | Section | Purpose |
|------|---------|---------|
| 2 | CUDA Configuration | Sets environment variables — **must run before any torch import** |
| 4 | PyTorch Validation | Confirms pre-installed PyTorch + CUDA are working |
| 6 | Library Imports | Imports core libraries; prints system info |
| 8 | GPU Validation | 4-step test: detection → alloc → matmul → cleanup |
| 10 | System Dependencies | Installs `ffmpeg`, `portaudio19-dev` via `apt-get` |
| 12 | AI Library Install | Installs `transformers`, `diffusers`, `mlflow`, `streamlit`, etc. (3–7 min) |
| 14 | Infrastructure Test | Verifies all library imports; prints version numbers; pass/fail summary |
| 16 | Hugging Face Auth | Guided token entry; saves credentials for gated-model downloads |
| 18 | Model Download | Downloads all 5 GGUF models to `/home/jovyan/local/` |
| 20 | Setup Summary | GPU info, auth status, model paths, next steps |
| 22 | Quick Reference | Code snippets for Zephyr, FLUX, Whisper, XTTS |

## After Setup — Starter Notebooks

| Notebook | Capability | Model Used |
|----------|-----------|------------|
| [chatbot-starter.ipynb](chatbot-starter.ipynb) | Conversational AI | Zephyr 7B Beta Q5_K_M |
| [document-analyzer-starter.ipynb](document-analyzer-starter.ipynb) | Document Q&A (RAG) | Llama 3.1 8B Q6_K_L |
| [image-gen-starter.ipynb](image-gen-starter.ipynb) | Text-to-Image | FLUX.1-dev Q4_K_S |
| [voice-assistant-starter.ipynb](voice-assistant-starter.ipynb) | Voice AI (STT + TTS) | Whisper V3 Turbo + XTTS v2 |

## 1. Environment Configuration

Set the required environment variables **before any `import torch` call**.
CUDA variables are read by the runtime at import time — they cannot be changed after the first GPU import.

### CUDA Variables
| Variable | Value | Why |
|----------|-------|-----|
| `CUDA_VISIBLE_DEVICES` | `"0"` | Restricts PyTorch to GPU 0; avoids accidental multi-GPU fragmentation |
| `PYTORCH_CUDA_ALLOC_CONF` | `"expandable_segments:True"` | Reduces memory fragmentation by letting the allocator grow segments dynamically |
| `CUDA_LAUNCH_BLOCKING` | `"0"` | Async kernel launches (set to `"1"` only when debugging CUDA errors) |

### Hugging Face Cache
| Variable | Value | Purpose |
|----------|-------|---------|
| `HF_HOME` | `local/data/huggingface` | Centralized Hugging Face cache — survives workspace restarts |
| `TRANSFORMERS_CACHE` | `local/data/huggingface/hub` | Transformer model cache directory |

> ⚠️ **Run this cell first** — before importing `torch` or any other GPU/AI library.

In [ ]:
import os
import sys
import time

# ── CUDA environment — must be set before any torch import ───────────────────

# Restrict execution to GPU device 0 (avoids accidental multi-GPU problems)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Allow PyTorch's memory allocator to grow segments dynamically.
# This significantly reduces OOM errors when loading large models.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Keep CUDA launches asynchronous for performance.
# Set to "1" only when debugging CUDA errors (shows exact failing line).
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")

# ── Hugging Face cache — persists across workspace restarts ──────────────────
os.environ.setdefault("HF_HOME", "local/data/huggingface")
os.environ.setdefault("TRANSFORMERS_CACHE", "local/data/huggingface/hub")

# ── Ensure project root is on the Python path so `src.*` imports work ─────────
sys.path.insert(0, "..")

start_time = time.time()

print("─" * 55)
print("  Environment Configuration")
print("─" * 55)
print("  CUDA:")
print(f"    CUDA_VISIBLE_DEVICES     : {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"    PYTORCH_CUDA_ALLOC_CONF  : {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"    CUDA_LAUNCH_BLOCKING     : {os.environ['CUDA_LAUNCH_BLOCKING']}")
print()
print("  Hugging Face Cache:")
print(f"    HF_HOME                  : {os.environ['HF_HOME']}")
print(f"    TRANSFORMERS_CACHE       : {os.environ['TRANSFORMERS_CACHE']}")
print("─" * 55)
print("✅ Cell 1 complete — environment variables configured")
print("⏱️  Setup notebook started")


───────────────────────────────────────────────────────
  Environment Configuration
───────────────────────────────────────────────────────
  CUDA:
    CUDA_VISIBLE_DEVICES     : 0
    PYTORCH_CUDA_ALLOC_CONF  : expandable_segments:True
    CUDA_LAUNCH_BLOCKING     : 0

  Hugging Face Cache:
    HF_HOME                  : /data/huggingface
    TRANSFORMERS_CACHE       : /data/huggingface/hub
───────────────────────────────────────────────────────
✅ Cell 1 complete — environment variables configured
⏱️  Setup notebook started


## 2. PyTorch Validation

Confirm that the pre-installed PyTorch build has CUDA support.

In [2]:
import torch

cuda_ok = torch.cuda.is_available()

print("─" * 55)
print("  PyTorch Validation (pre-installed)")
print("─" * 55)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {'✅ Available' if cuda_ok else '❌ Not found — GPU required'}")

if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {vram_gb:.1f} GB {'✅' if vram_gb >= 8 else '⚠️ (8 GB minimum recommended)'}")
    print(f"  Compute  : {props.major}.{props.minor}")

    # Minimum version gate — torch >= 2.8.0 required by this project
    parts = torch.__version__.split("+")[0].split(".")
    if int(parts[0]) < 2 or (int(parts[0]) == 2 and int(parts[1]) < 8):
        print(f"  ⚠️  torch {torch.__version__} is below minimum — this project requires torch ≥ 2.8.0")
    else:
        print(f"  ✅ torch {torch.__version__} meets minimum requirement (≥ 2.8.0)")
else:
    print("  ⚠️ GPU not detected. All models in this project require a CUDA-capable GPU.")

print("─" * 55)
print(f"✅ Cell 2 complete — PyTorch {torch.__version__} verified")

───────────────────────────────────────────────────────
  PyTorch Validation (pre-installed)
───────────────────────────────────────────────────────
  PyTorch  : 2.9.1+cu128
  CUDA     : ✅ Available
  GPU      : NVIDIA RTX A4000
  VRAM     : 17.2 GB ✅
  Compute  : 8.6
  ✅ torch 2.9.1+cu128 meets minimum requirement (≥ 2.8.0)
───────────────────────────────────────────────────────
✅ Cell 2 complete — PyTorch 2.9.1+cu128 verified


## 3. Library Imports

Confirm that the core Python standard library and PyTorch are importable and print
the runtime environment details.
This snapshot helps reproduce bugs: paste the output when asking for support.

In [3]:
import os
import sys
import platform
import torch

print("─" * 55)
print("  System Information")
print("─" * 55)
print(f"  Python   : {sys.version.split()[0]}")
print(f"  Platform : {platform.system()} {platform.release()}")
print(f"  Arch     : {platform.machine()}")
print()
print("  Core PyTorch")
print(f"  torch    : {torch.__version__}")
print(f"  CUDA     : {torch.version.cuda or 'None'}")
print(f"  cuDNN    : {torch.backends.cudnn.version() or 'None'}")
print("─" * 55)
print("✅ Cell 3 complete — core imports confirmed")

───────────────────────────────────────────────────────
  System Information
───────────────────────────────────────────────────────
  Python   : 3.12.7
  Platform : Linux 5.15.167.4-microsoft-standard-WSL2
  Arch     : x86_64

  Core PyTorch
  torch    : 2.9.1+cu128
  CUDA     : 12.8
  cuDNN    : 91002
───────────────────────────────────────────────────────
✅ Cell 3 complete — core imports confirmed


## 4. GPU Validation

Run a 4-step functional test to confirm your GPU is correctly bound to PyTorch and can
perform the operations that the AI models rely on.

| Step | Test | What it checks |
|------|------|---------------|
| 1 | CUDA detection | `torch.cuda.is_available()` returns `True` |
| 2 | Memory allocation | Can allocate a 4 MB tensor on GPU memory |
| 3 | Matrix multiply | Can run a 512×512 fp32 matmul on GPU |
| 4 | VRAM cleanup | `torch.cuda.empty_cache()` + `gc.collect()` succeed |

All 4 steps must pass before proceeding to model downloads.

In [4]:
import torch

results = []

# ── Step 1: CUDA detection ────────────────────────────────────────────────────
cuda_ok = torch.cuda.is_available()
results.append(("CUDA detection", cuda_ok,
                f"Device: {torch.cuda.get_device_name(0)}" if cuda_ok else "CUDA not found"))
print(f"[{'✅ PASS' if cuda_ok else '❌ FAIL'}] Step 1 — CUDA detection")

[✅ PASS] Step 1 — CUDA detection


## 5. System Dependencies

Install the system-level packages required by the AI libraries in this blueprint.
These are **OS packages** installed via `apt-get`, not Python packages.

| Package | Purpose | Required by |
|---------|---------|-------------|
| `ffmpeg` | Audio/video transcoding — decodes MP3, OGG, FLAC → WAV | Whisper (pywhispercpp), XTTS v2 (CoquiTTS) |
| `portaudio19-dev` | System audio driver for real-time capture/playback | `pyaudio` (Python binding for PortAudio) |

> **Note:** This project uses `hf_hub_download()` for all model downloads (HTTP-based), so `git-lfs` is not required.

> **First-run time:** ~30 seconds.

In [5]:
import subprocess

print("─" * 55)
print("  System Dependencies Install")
print("─" * 55)

# Install ffmpeg (audio transcoding for Whisper/TTS) and portaudio19-dev (PortAudio C headers for pyaudio).
# Note: git-lfs is NOT installed — this project uses hf_hub_download() which is HTTP-based and needs no git-lfs.
try:
    subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True, capture_output=True)
    subprocess.run(
        ["sudo", "apt-get", "install", "-y", "-qq", "ffmpeg", "portaudio19-dev"],
        check=True, capture_output=True
    )
    print("  ✅ Installed: ffmpeg, portaudio19-dev")
except Exception as e:
    print(f"  ⚠️  apt-get failed: {e}")
    print("       ffmpeg is required for audio processing (Whisper STT, XTTS TTS).")
    print("       portaudio19-dev is required by pyaudio (real-time microphone capture).")
    print("       If sudo is unavailable, these may already be present in the base image.")

# Verify ffmpeg is accessible
try:
    out = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    first_line = out.stdout.strip().split("\n")[0]
    print(f"  ✅ ffmpeg   : {first_line}")
except Exception:
    print("  ❌ ffmpeg   : not found — audio processing will be unavailable")

print("─" * 55)
print("✅ Cell 5 complete — system dependencies installed")


───────────────────────────────────────────────────────
  System Dependencies Install
───────────────────────────────────────────────────────
  ✅ Installed: ffmpeg, portaudio19-dev
  ✅ ffmpeg   : ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
───────────────────────────────────────────────────────
✅ Cell 5 complete — system dependencies installed


## 6. AI Library Install

Install all project AI dependencies from `requirements.txt`.

Three packages need special handling and are installed separately with fallback logic:

| Package | Issue | Strategy |
|---------|-------|----------|
| `pyaudio` | Requires `portaudio.h` C headers from Cell 5. | Attempted after system deps; failure is non-blocking (no starter notebook uses real-time mic capture). |
| `faiss-gpu-cu12` | No pre-built wheel for CUDA 12.8 + Python 3.12. | GPU install attempted first; automatic fallback to `faiss-cpu` if it fails. |
| `coqui-tts` | Deep dependency tree with incomplete Python 3.12 wheels. | Normal install first; retry with `--no-build-isolation` if it fails. |

> **First-run time:** 3–7 minutes.

In [ ]:
import subprocess
import sys

_py = sys.executable


def _pip(*args):
    """Run pip with the current interpreter; return (returncode, stderr)."""
    r = subprocess.run(
        [_py, "-m", "pip", "install", "--quiet"] + list(args),
        capture_output=True, text=True,
    )
    return r.returncode, r.stderr


print("─" * 55)
print("  Stage 1: torchvision + torchaudio (CUDA 12.8 / cu128)")
print("─" * 55)
_tv_ok = False
try:
    import torchvision, torchaudio
    print(f"  ✅ torchvision  : {torchvision.__version__} (already installed)")
    print(f"  ✅ torchaudio   : {torchaudio.__version__} (already installed)")
    _tv_ok = True
except ImportError:
    _rc, _err = _pip(
        "torchvision==0.24.1", "torchaudio==2.9.1",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    )
    if _rc == 0:
        for _mod in list(sys.modules.keys()):
            if _mod.startswith("torchvision") or _mod.startswith("torchaudio"):
                del sys.modules[_mod]
        try:
            import torchvision, torchaudio  # noqa: F811
            print(f"  ✅ torchvision  : {torchvision.__version__}")
            print(f"  ✅ torchaudio   : {torchaudio.__version__}")
            _tv_ok = True
        except Exception as e:
            print(f"  ⚠️  Installed but import failed: {str(e)[:60]}...")
            print(f"  🔄 Restart kernel (Kernel → Restart) and re-run from Cell 1")
            _tv_ok = True
    else:
        print(f"  ❌ torchvision/torchaudio install failed (rc={_rc})")
        print(f"       {_err[-200:].strip()}")

# ── Stage 2: Core requirements (fragile packages excluded) ────────────────────
print()
print("─" * 55)
print("  Stage 2: Core requirements")
print("─" * 55)

with open("../requirements.txt") as _f:
    _safe = [
        l.strip() for l in _f
        if l.strip()
        and not l.strip().startswith("#")
        and "faiss-gpu-cu12" not in l
        and "coqui-tts" not in l
        and "pyaudio" not in l
    ]

_rc, _err = _pip(*_safe)
if _rc == 0:
    print("  ✅ Core requirements installed")
else:
    print(f"  ⚠️  Some packages had issues (rc={_rc}). Continuing.")
    print(f"       {_err[-200:].strip()}")

# ── Stage 3: pyaudio (requires portaudio19-dev from Cell 5) ──────────────────
print()
print("─" * 55)
print("  Stage 3: pyaudio")
print("─" * 55)
_pyaudio_ok = False
try:
    import pyaudio as _pa  # noqa: F401
    print("  ✅ pyaudio already installed")
    _pyaudio_ok = True
except ImportError:
    _rc, _ = _pip("pyaudio")
    if _rc == 0:
        print("  ✅ pyaudio installed")
        _pyaudio_ok = True
    else:
        print("  ⚠️  pyaudio failed — real-time mic capture unavailable")
        print("       Ensure Cell 5 ran successfully, then re-run this cell.")
        print("       No starter notebook requires real-time audio (voice demo uses file upload).")

# ── Stage 4: FAISS — GPU preferred, CPU fallback ─────────────────────────────
print()
print("─" * 55)
print("  Stage 4: FAISS (GPU preferred → CPU fallback)")
print("─" * 55)
_faiss_gpu = False
try:
    import faiss as _f
    _faiss_gpu = hasattr(_f, "StandardGpuResources")
    print(f"  ✅ faiss already installed ({'GPU' if _faiss_gpu else 'CPU'})")
except ImportError:
    _rc, _ = _pip("faiss-gpu-cu12")
    if _rc == 0:
        print("  ✅ faiss-gpu-cu12 installed")
        _faiss_gpu = True
    else:
        print("  ⚠️  faiss-gpu-cu12: no compatible wheel for CUDA 12.8 / Python 3.12")
        _rc2, _ = _pip("faiss-cpu")
        if _rc2 == 0:
            print("  ✅ faiss-cpu installed (CPU fallback — functionally identical at educational scale)")
        else:
            print("  ❌ faiss-cpu also failed — check pip output above")

# ── Stage 5: coqui-tts — retry with --no-build-isolation if needed ───────────
print()
print("─" * 55)
print("  Stage 5: coqui-tts (XTTS v2 voice synthesis)")
print("─" * 55)
_tts_ok = False
try:
    from TTS.api import TTS as _T  # noqa: F401
    print("  ✅ coqui-tts already installed")
    _tts_ok = True
except ImportError:
    _rc, _ = _pip("coqui-tts")
    if _rc == 0:
        print("  ✅ coqui-tts installed")
        _tts_ok = True
    else:
        print("  ⚠️  Standard install failed — retrying with --no-build-isolation ...")
        _rc2, _ = _pip("--no-build-isolation", "coqui-tts")
        if _rc2 == 0:
            print("  ✅ coqui-tts installed (--no-build-isolation)")
            _tts_ok = True
        else:
            print("  ❌ coqui-tts could not be installed — voice TTS (XTTS v2) disabled")
            print("       Voice STT (pywhispercpp) and all other capabilities still work.")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("─" * 55)
print(f"  torchvision/torchaudio: {'installed' if _tv_ok else 'UNAVAILABLE'}")
print(f"  FAISS   : {'GPU (faiss-gpu-cu12)' if _faiss_gpu else 'CPU fallback (faiss-cpu)'}")
print(f"  CoquiTTS: {'installed' if _tts_ok else 'UNAVAILABLE — voice TTS disabled'}")
print(f"  pyaudio : {'installed' if _pyaudio_ok else 'UNAVAILABLE — real-time mic disabled (no starter notebook affected)'}")
print("─" * 55)
print("✅ Cell 6 complete — AI libraries installed")


───────────────────────────────────────────────────────
  Stage 1: torchvision + torchaudio (CUDA 12.8 / cu128)
───────────────────────────────────────────────────────
  ✅ torchvision  : 0.24.1+cu128 (already installed)
  ✅ torchaudio   : 2.9.1+cu128 (already installed)

───────────────────────────────────────────────────────
  Stage 2: Core requirements
───────────────────────────────────────────────────────
  ✅ Core requirements installed

───────────────────────────────────────────────────────
  Stage 3: pyaudio
───────────────────────────────────────────────────────
  ✅ pyaudio already installed

───────────────────────────────────────────────────────
  Stage 4: FAISS (GPU preferred → CPU fallback)
───────────────────────────────────────────────────────
  ✅ faiss already installed (GPU)

───────────────────────────────────────────────────────
  Stage 5: coqui-tts (XTTS v2 voice synthesis)
───────────────────────────────────────────────────────


/opt/conda/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


  ✅ coqui-tts installed

───────────────────────────────────────────────────────
  torchvision/torchaudio: installed
  FAISS   : GPU (faiss-gpu-cu12)
  CoquiTTS: installed
  pyaudio : installed
───────────────────────────────────────────────────────
✅ Cell 6 complete — AI libraries installed


## 7. Infrastructure Test

Import every critical AI library and confirm its version number.
A 5/5 pass means the environment is fully wired and all starter notebooks can run.

In [7]:
total = 0
passed = 0

print("═" * 60)
print("  Infrastructure Test — AI Libraries")
print("═" * 60)

# ── 4.2.1 Core ML/AI Frameworks ──────────────────────────────────────────────
print("\n  4.2.1 — Core ML/AI Frameworks")
print("  " + "─" * 56)

try:
    import transformers
    print(f"  ✅ {'transformers':<32} {transformers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'transformers':<32} {str(e)[:48]}")
total += 1

try:
    import diffusers
    print(f"  ✅ {'diffusers':<32} {diffusers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'diffusers':<32} {str(e)[:48]}")
total += 1

try:
    import accelerate
    print(f"  ✅ {'accelerate':<32} {accelerate.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'accelerate':<32} {str(e)[:48]}")
total += 1

try:
    import bitsandbytes
    print(f"  ✅ {'bitsandbytes':<32} {bitsandbytes.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'bitsandbytes':<32} {str(e)[:48]}")
total += 1

try:
    import safetensors
    print(f"  ✅ {'safetensors':<32} {safetensors.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'safetensors':<32} {str(e)[:48]}")
total += 1

try:
    import sentencepiece
    print(f"  ✅ {'sentencepiece':<32} {sentencepiece.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'sentencepiece':<32} {str(e)[:48]}")
total += 1

# ── 4.2.2 Interface & Deployment ─────────────────────────────────────────────
print("\n  4.2.2 — Interface & Deployment")
print("  " + "─" * 56)

try:
    import mlflow
    print(f"  ✅ {'mlflow':<32} {mlflow.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'mlflow':<32} {str(e)[:48]}")
total += 1

try:
    import huggingface_hub
    print(f"  ✅ {'huggingface_hub':<32} {huggingface_hub.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'huggingface_hub':<32} {str(e)[:48]}")
total += 1

try:
    import fastapi
    print(f"  ✅ {'fastapi':<32} {fastapi.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'fastapi':<32} {str(e)[:48]}")
total += 1

try:
    import pydantic
    print(f"  ✅ {'pydantic':<32} {pydantic.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pydantic':<32} {str(e)[:48]}")
total += 1

try:
    import datasets
    print(f"  ✅ {'datasets':<32} {datasets.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'datasets':<32} {str(e)[:48]}")
total += 1

try:
    import streamlit
    print(f"  ✅ {'streamlit':<32} {streamlit.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'streamlit':<32} {str(e)[:48]}")
total += 1

# ── 4.2.3 Document & Media Processing ────────────────────────────────────────
print("\n  4.2.3 — Document & Media Processing")
print("  " + "─" * 56)

try:
    import pypdf
    print(f"  ✅ {'pypdf':<32} {pypdf.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pypdf':<32} {str(e)[:48]}")
total += 1

try:
    import fitz
    print(f"  ✅ {'pymupdf (fitz)':<32} {fitz.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pymupdf (fitz)':<32} {str(e)[:48]}")
total += 1

try:
    from imwatermark import WatermarkEncoder  # noqa: F401
    print(f"  ✅ {'invisible_watermark':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'invisible_watermark':<32} {str(e)[:48]}")
total += 1

try:
    from PIL import Image  # noqa: F401
    print(f"  ✅ {'PIL (pillow)':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'PIL (pillow)':<32} {str(e)[:48]}")
total += 1

try:
    import cv2
    print(f"  ✅ {'cv2 (opencv)':<32} {cv2.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'cv2 (opencv)':<32} {str(e)[:48]}")
total += 1

# ── 4.2.4 Audio & Voice Processing ───────────────────────────────────────────
print("\n  4.2.4 — Audio & Voice Processing")
print("  " + "─" * 56)

try:
    from pywhispercpp.model import Model as _W  # noqa: F401
    print(f"  ✅ {'pywhispercpp (voice STT)':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'pywhispercpp (voice STT)':<32} {str(e)[:48]}")
total += 1

try:
    import torchaudio
    print(f"  ✅ {'torchaudio':<32} {torchaudio.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'torchaudio':<32} {str(e)[:48]}")
total += 1

try:
    import librosa
    print(f"  ✅ {'librosa':<32} {librosa.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'librosa':<32} {str(e)[:48]}")
total += 1

try:
    import soundfile
    print(f"  ✅ {'soundfile':<32} {soundfile.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'soundfile':<32} {str(e)[:48]}")
total += 1

try:
    import sounddevice
    print(f"  ✅ {'sounddevice':<32} {sounddevice.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'sounddevice':<32} {str(e)[:48]}")
total += 1

# ── 4.2.5 Agentic AI & RAG ───────────────────────────────────────────────────
print("\n  4.2.5 — Agentic AI & RAG")
print("  " + "─" * 56)

try:
    import langchain
    print(f"  ✅ {'langchain':<32} {langchain.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'langchain':<32} {str(e)[:48]}")
total += 1

try:
    from langchain_community.llms import LlamaCpp  # noqa: F401
    print(f"  ✅ {'langchain_community (LlamaCpp)':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'langchain_community (LlamaCpp)':<32} {str(e)[:48]}")
total += 1

try:
    import chromadb
    print(f"  ✅ {'chromadb':<32} {chromadb.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'chromadb':<32} {str(e)[:48]}")
total += 1

try:
    import sentence_transformers
    print(f"  ✅ {'sentence_transformers':<32} {sentence_transformers.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'sentence_transformers':<32} {str(e)[:48]}")
total += 1

# ── 4.2.6 Monitoring & Utilities ─────────────────────────────────────────────
print("\n  4.2.6 — Monitoring & Utilities")
print("  " + "─" * 56)

try:
    import psutil
    print(f"  ✅ {'psutil':<32} {psutil.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'psutil':<32} {str(e)[:48]}")
total += 1

try:
    import GPUtil  # noqa: F401
    print(f"  ✅ {'gputil':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'gputil':<32} {str(e)[:48]}")
total += 1

try:
    import plotly
    print(f"  ✅ {'plotly':<32} {plotly.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'plotly':<32} {str(e)[:48]}")
total += 1

try:
    import yaml
    print(f"  ✅ {'pyyaml':<32} {yaml.__version__}"); passed += 1
except Exception as e:
    print(f"  ❌ {'pyyaml':<32} {str(e)[:48]}")
total += 1

try:
    from dotenv import load_dotenv  # noqa: F401
    print(f"  ✅ {'python-dotenv':<32} (installed)"); passed += 1
except Exception as e:
    print(f"  ❌ {'python-dotenv':<32} {str(e)[:48]}")
total += 1

# ── FAISS — GPU or CPU both acceptable ───────────────────────────────────────
print("\n  FAISS")
print("  " + "─" * 56)
total += 1
try:
    import faiss
    _is_gpu = hasattr(faiss, "StandardGpuResources")
    passed += 1
    if _is_gpu:
        print("  ✅ faiss-gpu-cu12                GPU-accelerated")
    else:
        print("  ✅ faiss-cpu                     CPU fallback (functional for this project)")
except ImportError as e:
    print(f"  ❌ faiss                          {str(e)[:48]}")
    print("       Re-run Cell 6 to retry the faiss install.")

# ── CoquiTTS — required for voice TTS; STT is independent ────────────────────
print("\n  CoquiTTS / XTTS v2 (voice TTS)")
print("  " + "─" * 56)
total += 1
try:
    from TTS.api import TTS as _TTS  # noqa: F401
    passed += 1
    print("  ✅ TTS (CoquiTTS)                (installed)")
except ImportError:
    print("  ❌ TTS (CoquiTTS)                not installed")
    print("       Voice TTS (XTTS v2) unavailable. Voice STT (pywhispercpp) still works.")

# ── pyaudio — non-blocking ───────────────────────────────────────────────────
print("\n  pyaudio (real-time microphone capture)")
print("  " + "─" * 56)
try:
    import pyaudio  # noqa: F401
    print("  ✅ pyaudio                       (installed)")
except ImportError:
    print("  ⚠️  pyaudio                       not installed  [non-blocking]")
    print("       No starter notebook requires real-time mic capture (voice demo uses file upload).")

print("\n" + "═" * 60)
print(f"  Grand Total: {passed}/{total} libraries available")
print("═" * 60)

if passed == total:
    print("✅ Cell 7 complete — all libraries verified")
else:
    failed = total - passed
    print(f"⚠️  {failed} library/libraries not available — re-run Cell 6 if unexpected")
    print("   Note: faiss-cpu, coqui-tts, or pyaudio failures do NOT block any starter notebook.")


════════════════════════════════════════════════════════════
  Infrastructure Test — AI Libraries
════════════════════════════════════════════════════════════

  4.2.1 — Core ML/AI Frameworks
  ────────────────────────────────────────────────────────
  ✅ transformers                     4.57.6
  ✅ diffusers                        0.36.0
  ✅ accelerate                       1.12.0
  ✅ bitsandbytes                     0.49.2
  ✅ safetensors                      0.6.2
  ✅ sentencepiece                    0.2.1

  4.2.2 — Interface & Deployment
  ────────────────────────────────────────────────────────
  ✅ mlflow                           3.1.0
  ✅ huggingface_hub                  0.36.2
  ✅ fastapi                          0.121.0
  ✅ pydantic                         2.12.4
  ✅ datasets                         3.0.1
  ✅ streamlit                        1.54.0

  4.2.3 — Document & Media Processing
  ────────────────────────────────────────────────────────
  ✅ pypdf                        

2026-02-23 16:35:20.890794335 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


  ✅ chromadb                         0.5.20
  ✅ sentence_transformers            3.0.0

  4.2.6 — Monitoring & Utilities
  ────────────────────────────────────────────────────────
  ✅ psutil                           7.1.3
  ✅ gputil                           (installed)
  ✅ plotly                           6.5.2
  ✅ pyyaml                           6.0.3
  ✅ python-dotenv                    (installed)

  FAISS
  ────────────────────────────────────────────────────────
  ✅ faiss-gpu-cu12                GPU-accelerated

  CoquiTTS / XTTS v2 (voice TTS)
  ────────────────────────────────────────────────────────
  ❌ TTS (CoquiTTS)                not installed
       Voice TTS (XTTS v2) unavailable. Voice STT (pywhispercpp) still works.

  pyaudio (real-time microphone capture)
  ────────────────────────────────────────────────────────
  ✅ pyaudio                       (installed)

════════════════════════════════════════════════════════════
  Grand Total: 32/33 libraries available
══════

## 8. Hugging Face Authentication

Some models in this project require a **free Hugging Face account** and an **access token**.

### Steps to get your token:
1. Create a free account at [huggingface.co](https://huggingface.co/join)
2. Go to **Settings → Access Tokens** → [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Click **"New token"** → Name it (e.g., `ai-studio`) → select **"Read"** → **"Generate token"**
4. Copy the token — it starts with `hf_...`

### For FLUX.1-dev (gated model):
FLUX.1-dev requires accepting the license before download:
5. Visit [https://huggingface.co/black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
6. Click **"Access repository"** and accept the license agreement
7. Wait a few minutes for your access to be approved (usually instant)

> **Security note:** Your token is entered via `getpass` — it will not appear on screen
> and will not be saved to the notebook file.

In [8]:
import getpass
from huggingface_hub import login, whoami

hf_token = getpass.getpass("🔑 Paste your Hugging Face token (hidden): ")

try:
    # login() caches the token in ~/.cache/huggingface/token
    # add_to_git_credential=False keeps it out of git credential storage
    login(token=hf_token, add_to_git_credential=False)
    user = whoami()
    print(f"✅ Authenticated as : {user['name']}")
    print(f"   Account type     : {user.get('type', 'user')}")
    print(f"   Email            : {user.get('email', '(not set)')}")
    hf_auth_ok = True
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("   Check your token at https://huggingface.co/settings/tokens")
    hf_auth_ok = False

print()
print("✅ Cell 8 complete" if hf_auth_ok else "⚠️  Fix auth before model downloads")

🔑 Paste your Hugging Face token (hidden):  ········


❌ Authentication failed: [Errno 13] Permission denied: '/data'
   Check your token at https://huggingface.co/settings/tokens

⚠️  Fix auth before model downloads


## 9. Model Download

Download all quantized GGUF models used by the four starter notebooks.
Each download is **skipped automatically** if the file already exists — re-running this
cell is safe and will only fetch what is missing.

> ⚠️ **Total download size:** ~40 GB. Ensure you have sufficient disk space at
> `/home/jovyan/local/` before proceeding.

> **Note on FLUX.1-dev:** This model is gated. You must have accepted the license
> at [https://huggingface.co/black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
> and completed Cell 8 (HF Auth) before this download will succeed.

| # | Capability | Model | Repo | Size |
|---|-----------|-------|------|------|
| 1 | Chatbot | Zephyr 7B Beta Q5_K_M | `TheBloke/zephyr-7B-beta-GGUF` | ~4.8 GB |
| 2 | Document + Voice LLM | Llama 3.1 8B Q6_K_L | `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF` | ~6.6 GB |
| 3a | Image Gen (GGUF transformer) | FLUX.1-dev Q4_K_S | `city96/FLUX.1-dev-gguf` | ~6.9 GB |
| 3b | Image Gen (pipeline components) | FLUX.1-dev encoders + VAE | `black-forest-labs/FLUX.1-dev` | ~22 GB |
| 4 | Voice STT | Whisper Large V3 Turbo Q4_1 | `xkeyC/whisper-large-v3-turbo-gguf` | ~0.5 GB |
| 5 | Voice TTS | XTTS v2 F16 | `GenMedLabs/xtts-gguf` | ~2.8 GB |

In [9]:
import os
import time
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

# ── Base directory for all locally stored models ──────────────────────────────
# This project stores models at /home/jovyan/local/ (not datafabric) so that
# students can download and own their model files directly in the JupyterHub home.
LOCAL_BASE = Path("/home/jovyan/local")
LOCAL_BASE.mkdir(parents=True, exist_ok=True)

download_results = []


def _skip_or_download(label: str, check_path: Path, download_fn) -> bool:
    """Download a model asset unless it already exists at check_path."""
    if check_path.exists():
        size_gb = (
            sum(f.stat().st_size for f in check_path.rglob("*") if f.is_file()) / 1e9
            if check_path.is_dir()
            else check_path.stat().st_size / 1e9
        )
        print(f"  ⏭️  {label}")
        print(f"       → already at {check_path} ({size_gb:.2f} GB)")
        return True
    print(f"  ⬇️  {label}")
    print(f"       → downloading to {check_path} ...")
    t0 = time.time()
    try:
        download_fn()
        elapsed = time.time() - t0
        size_gb = (
            sum(f.stat().st_size for f in check_path.rglob("*") if f.is_file()) / 1e9
            if check_path.is_dir()
            else check_path.stat().st_size / 1e9
        )
        print(f"       ✅ done in {elapsed:.0f}s ({size_gb:.2f} GB)")
        return True
    except Exception as e:
        print(f"       ❌ FAILED: {e}")
        return False


print("─" * 60)
print("  Model Download  —  /home/jovyan/local/")
print("─" * 60)

# ── 1. Chatbot LLM: Zephyr 7B Beta Q5_K_M GGUF ───────────────────────────────
#    Zephyr is a fine-tuned version of Mistral 7B optimised for instruction following.
#    Q5_K_M = 5-bit K-quantization (medium variant) — excellent quality/size trade-off.
dest_dir  = LOCAL_BASE / "zephyr-7b-beta"
dest_file = dest_dir / "zephyr-7b-beta.Q5_K_M.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Chatbot LLM  — Zephyr 7B Beta Q5_K_M  (~4.8 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="TheBloke/zephyr-7B-beta-GGUF",
        filename="zephyr-7b-beta.Q5_K_M.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Chatbot LLM  (Zephyr 7B Beta Q5_K_M)", ok, str(dest_file)))

# ── 2. Document + Voice LLM: Meta-Llama 3.1 8B Q6_K_L GGUF ──────────────────
#    Shared between the document analyzer and voice assistant to save disk space.
#    Q6_K_L = 6-bit K-quantization (large variant) — near full-precision quality.
dest_dir  = LOCAL_BASE / "meta-llama3.1-8b-Q6"
dest_file = dest_dir / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Document + Voice LLM  — Llama 3.1 8B Q6_K_L  (~6.6 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        filename="Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Document + Voice LLM  (Llama 3.1 8B Q6_K_L)", ok, str(dest_file)))

# ── 3a. Image Gen: FLUX.1-dev GGUF transformer ────────────────────────────────
#    Q4_K_S = 4-bit K-quantization (small) — best size for GPU inference.
#    This file replaces only the transformer block in the FLUX.1-dev pipeline.
dest_dir  = LOCAL_BASE / "flux1-dev"
gguf_file = dest_dir / "flux1-dev-Q4_K_S.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok_transformer = _skip_or_download(
    "Image Gen GGUF transformer  — FLUX.1-dev Q4_K_S  (~6.9 GB)",
    gguf_file,
    lambda: hf_hub_download(
        repo_id="city96/FLUX.1-dev-gguf",
        filename="flux1-dev-Q4_K_S.gguf",
        local_dir=str(dest_dir),
    ),
)

# ── 3b. Image Gen: FLUX.1-dev pipeline components ─────────────────────────────
#    Downloads text encoders (T5 + CLIP), VAE, scheduler, and tokenizers.
#    The transformer weights are excluded (we use the GGUF file from step 3a).
#    ⚠️  REQUIRES: HF auth + accepted license at black-forest-labs/FLUX.1-dev
config_marker = dest_dir / "model_index.json"
ok_pipeline = _skip_or_download(
    "Image Gen pipeline components  — FLUX.1-dev encoders + VAE  (~22 GB)",
    config_marker,
    lambda: snapshot_download(
        repo_id="black-forest-labs/FLUX.1-dev",
        local_dir=str(dest_dir),
        # Exclude the large non-GGUF transformer — we use the GGUF file above instead
        ignore_patterns=["transformer/*", "*.bin"],
    ),
)
ok = ok_transformer and ok_pipeline
download_results.append(("Image Gen  (FLUX.1-dev GGUF Q4_K_S)", ok, str(dest_dir)))

# ── 4. Voice STT: Whisper Large V3 Turbo GGUF ─────────────────────────────────
#    Whisper Large V3 Turbo is a streamlined Whisper variant — faster than V3
#    with minimal quality loss. Loaded via pywhispercpp (whisper.cpp Python binding).
#    Q4_1 = 4-bit quantization, approximately 0.5 GB.
dest_dir  = LOCAL_BASE / "whisper-large-v3-turbo"
dest_file = dest_dir / "model_q4_1.gguf"
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Voice STT  — Whisper Large V3 Turbo Q4_1  (~0.5 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="xkeyC/whisper-large-v3-turbo-gguf",
        filename="model_q4_1.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Voice STT  (Whisper V3 Turbo Q4_1 GGUF)", ok, str(dest_file)))

# ── 5. Voice TTS: XTTS v2 F16 GGUF ───────────────────────────────────────────
#    XTTS v2 is Coqui's multilingual zero-shot text-to-speech model.
#    F16 = float16 full-precision weights in GGUF container format (~2.8 GB).
#    Loaded via the TTS (CoquiTTS) Python library.
dest_dir  = LOCAL_BASE / "xtts-v2"
dest_file = dest_dir / "gguf" / "xtts_v2_f16.gguf"   # mirrors the repo path
dest_dir.mkdir(parents=True, exist_ok=True)
ok = _skip_or_download(
    "Voice TTS  — XTTS v2 F16 GGUF  (~2.8 GB)",
    dest_file,
    lambda: hf_hub_download(
        repo_id="GenMedLabs/xtts-gguf",
        filename="gguf/xtts_v2_f16.gguf",
        local_dir=str(dest_dir),
    ),
)
download_results.append(("Voice TTS  (XTTS v2 F16 GGUF)", ok, str(dest_file)))

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Download Summary")
print("─" * 60)
all_ok = True
for label, status, path in download_results:
    icon = "✅" if status else "❌"
    print(f"  {icon} {label}")
    print(f"       {path}")
    if not status:
        all_ok = False
print("─" * 60)
if all_ok:
    print("✅ Cell 9 complete — all models ready")
else:
    print("⚠️  Some downloads failed. Check errors above and re-run.")

────────────────────────────────────────────────────────────
  Model Download  —  /home/jovyan/local/
────────────────────────────────────────────────────────────
  ⬇️  Chatbot LLM  — Zephyr 7B Beta Q5_K_M  (~4.8 GB)
       → downloading to /home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf ...
{"timestamp":"2026-02-23T16:35:47.112442Z","level":"ERROR","fields":{"message":"Error logging to file \"/data/huggingface/xet/logs/xet_20260223T163547108+0000_5490.log\" (Permission denied (os error 13)); falling back to console logging."},"filename":"/home/runner/work/xet-core/xet-core/xet_logging/src/logging.rs","line_number":58}


zephyr-7b-beta.Q5_K_M.gguf:   0%|          | 0.00/5.13G [00:00<?, ?B/s]

       ❌ FAILED: Data processing error: I/O error: Permission denied (os error 13)
  ⬇️  Document + Voice LLM  — Llama 3.1 8B Q6_K_L  (~6.6 GB)
       → downloading to /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf ...


Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

       ❌ FAILED: Data processing error: I/O error: Permission denied (os error 13)
  ⬇️  Image Gen GGUF transformer  — FLUX.1-dev Q4_K_S  (~6.9 GB)
       → downloading to /home/jovyan/local/flux1-dev/flux1-dev-Q4_K_S.gguf ...


...

       ❌ FAILED: 401 Client Error. (Request ID: Root=1-699c8201-760813d712a8326e19e31f93;2ea7a609-c730-44ab-9cff-217e38f5a3ec)

Cannot access gated repo for url https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21/.gitattributes.
Access to model black-forest-labs/FLUX.1-dev is restricted. You must have access to it and be authenticated to access it. Please log in.
  ⬇️  Voice STT  — Whisper Large V3 Turbo Q4_1  (~0.5 GB)
       → downloading to /home/jovyan/local/whisper-large-v3-turbo/model_q4_1.gguf ...


model_q4_1.gguf:   0%|          | 0.00/526M [00:00<?, ?B/s]

       ❌ FAILED: Data processing error: I/O error: Permission denied (os error 13)
  ⬇️  Voice TTS  — XTTS v2 F16 GGUF  (~2.8 GB)
       → downloading to /home/jovyan/local/xtts-v2/gguf/xtts_v2_f16.gguf ...


xtts_v2_f16.gguf:   0%|          | 0.00/395 [00:00<?, ?B/s]

       ✅ done in 0s (0.00 GB)

────────────────────────────────────────────────────────────
  Download Summary
────────────────────────────────────────────────────────────
  ❌ Chatbot LLM  (Zephyr 7B Beta Q5_K_M)
       /home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf
  ❌ Document + Voice LLM  (Llama 3.1 8B Q6_K_L)
       /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
  ❌ Image Gen  (FLUX.1-dev GGUF Q4_K_S)
       /home/jovyan/local/flux1-dev
  ❌ Voice STT  (Whisper V3 Turbo Q4_1 GGUF)
       /home/jovyan/local/whisper-large-v3-turbo/model_q4_1.gguf
  ✅ Voice TTS  (XTTS v2 F16 GGUF)
       /home/jovyan/local/xtts-v2/gguf/xtts_v2_f16.gguf
────────────────────────────────────────────────────────────
⚠️  Some downloads failed. Check errors above and re-run.


## 10. Setup Summary

Full status report of the environment after completing all cells above.

In [10]:
import gc
import torch
from pathlib import Path

LOCAL_BASE = Path("/home/jovyan/local")

# ── GPU ───────────────────────────────────────────────────────────────────────
print("─" * 60)
print("  GPU Status")
print("─" * 60)
if torch.cuda.is_available():
    props  = torch.cuda.get_device_properties(0)
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {props.total_memory / 1e9:.1f} GB total  |  {free_b / 1e9:.1f} GB free")
    print(f"  Compute  : {props.major}.{props.minor}")
    print(f"  Driver   : CUDA {torch.version.cuda}")
else:
    print("  ❌ GPU not available")

# ── HF Auth ───────────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Hugging Face Auth")
print("─" * 60)
try:
    from huggingface_hub import whoami
    user = whoami()
    print(f"  ✅ Logged in as: {user['name']}")
except Exception:
    print("  ❌ Not authenticated — re-run Cell 8")

# ── Model Files ───────────────────────────────────────────────────────────────
print()
print("─" * 60)
print("  Model File Status")
print("─" * 60)
_model_checks = [
    ("Chatbot LLM  Zephyr 7B Beta Q5_K_M",
     LOCAL_BASE / "zephyr-7b-beta" / "zephyr-7b-beta.Q5_K_M.gguf"),
    ("Document LLM  Llama 3.1 8B Q6_K_L",
     LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"),
    ("Voice LLM  Llama 3.1 8B Q6_K_L  (shared)",
     LOCAL_BASE / "meta-llama3.1-8b-Q6" / "Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf"),
    ("Image Gen  FLUX.1-dev GGUF transformer",
     LOCAL_BASE / "flux1-dev" / "flux1-dev-Q4_K_S.gguf"),
    ("Image Gen  FLUX.1-dev pipeline",
     LOCAL_BASE / "flux1-dev" / "model_index.json"),
    ("Voice STT  Whisper V3 Turbo Q4_1",
     LOCAL_BASE / "whisper-large-v3-turbo" / "model_q4_1.gguf"),
    ("Voice TTS  XTTS v2 F16",
     LOCAL_BASE / "xtts-v2" / "gguf" / "xtts_v2_f16.gguf"),
]
all_present = True
for label, path in _model_checks:
    exists = path.exists()
    icon   = "✅" if exists else "❌"
    size   = f"({path.stat().st_size / 1e9:.2f} GB)" if exists and path.is_file() else ""
    print(f"  {icon} {label} {size}")
    print(f"       {path}")
    if not exists:
        all_present = False

# ── Next Steps ────────────────────────────────────────────────────────────────
elapsed = time.time() - start_time
print()
print("─" * 60)
print(f"  Total setup time: {elapsed:.0f}s  ({elapsed / 60:.1f} min)")
print("─" * 60)
print()
if all_present:
    print("🎉 Setup complete! Open a starter notebook to begin:")
    print()
    print("  📂 chatbot-starter.ipynb          ← Conversational AI  (Zephyr 7B)")
    print("  📂 document-analyzer-starter.ipynb ← Document Q&A  (Llama 3.1 8B)")
    print("  📂 image-gen-starter.ipynb         ← Text-to-Image  (FLUX.1-dev)")
    print("  📂 voice-assistant-starter.ipynb   ← Voice AI  (Whisper + XTTS v2)")
else:
    print("⚠️  Some models are still missing — re-run Cell 9 to retry downloads.")

────────────────────────────────────────────────────────────
  GPU Status
────────────────────────────────────────────────────────────
  GPU      : NVIDIA RTX A4000
  VRAM     : 17.2 GB total  |  16.0 GB free
  Compute  : 8.6
  Driver   : CUDA 12.8

────────────────────────────────────────────────────────────
  Hugging Face Auth
────────────────────────────────────────────────────────────
  ❌ Not authenticated — re-run Cell 8

────────────────────────────────────────────────────────────
  Model File Status
────────────────────────────────────────────────────────────
  ❌ Chatbot LLM  Zephyr 7B Beta Q5_K_M 
       /home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf
  ❌ Document LLM  Llama 3.1 8B Q6_K_L 
       /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
  ❌ Voice LLM  Llama 3.1 8B Q6_K_L  (shared) 
       /home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf
  ❌ Image Gen  FLUX.1-dev GGUF transformer 
       /home/jovyan/

## 11. Quick Reference

Common code snippets for working with the downloaded models in the starter notebooks.

### Chatbot — Zephyr 7B Beta (LlamaCpp)
```python
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/home/jovyan/local/zephyr-7b-beta/zephyr-7b-beta.Q5_K_M.gguf",
    n_gpu_layers=-1, n_ctx=4096, temperature=0.7,
)

# Zephyr prompt template (ChatML-like)
prompt = """<|system|>
You are a helpful assistant.</s>
<|user|>
What is the Eiffel Tower?</s>
<|assistant|>
"""
response = llm(prompt)
```

### Document Analyzer + Voice LLM — Llama 3.1 8B (LlamaCpp)
```python
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/home/jovyan/local/meta-llama3.1-8b-Q6/Meta-Llama-3.1-8B-Instruct-Q6_K_L.gguf",
    n_gpu_layers=-1, n_ctx=8192, temperature=0.0,
)

# Llama 3.1 prompt template
prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
Summarise this text.<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
```

### Image Generation — FLUX.1-dev GGUF (diffusers)
```python
import torch
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers.utils import GGUFQuantizationConfig

MODEL_DIR = "/home/jovyan/local/flux1-dev"
transformer = FluxTransformer2DModel.from_single_file(
    f"{MODEL_DIR}/flux1-dev-Q4_K_S.gguf",
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
)
pipe = FluxPipeline.from_pretrained(MODEL_DIR, transformer=transformer, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
image = pipe("A red cat on a spaceship", num_inference_steps=28, guidance_scale=3.5).images[0]
```

### Voice STT — Whisper V3 Turbo GGUF (pywhispercpp)
```python
from pywhispercpp.model import Model as WhisperCppModel

stt = WhisperCppModel("/home/jovyan/local/whisper-large-v3-turbo/model_q4_1.gguf")
segments = stt.transcribe("/path/to/audio.wav")
transcription = " ".join(seg.text.strip() for seg in segments)
```

### Voice TTS — XTTS v2 (TTS / CoquiTTS)
```python
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
tts.tts_to_file(text="Hello from AI Studio!", speaker_wav="ref.wav",
                language="en", file_path="output.wav")
```

### Common Error Fixes
| Error | Fix |
|-------|-----|
| `CUDA out of memory` | `torch.cuda.empty_cache(); import gc; gc.collect()` |
| `Model not found` | Re-run Cell 9 (Model Download) |
| `401 Unauthorized` | Re-run Cell 8 (HF Auth) |
| `FLUX access denied` | Accept license at huggingface.co/black-forest-labs/FLUX.1-dev |
| `Import error` | Re-run Cell 6 (AI Library Install) then kernel restart |